In [1]:
import pandas as pd
from pathlib import Path

PROJETO = Path.cwd().resolve().parent
PASTA_RAW = PROJETO / "data" / "raw"
PASTA_REPORTS = PROJETO / "reports"

PASTA_REPORTS.mkdir(parents=True, exist_ok=True)

df_avaliacao = pd.read_csv(
    PASTA_RAW / "ANALITICA_avaliacao_geral_sem_abertas.csv",
    dtype=str,
    encoding="utf-8-sig"
)

df_perfis = pd.read_csv(
    PASTA_RAW / "ANALITICA_perfis_harmonizados_sem_abertas.csv",
    dtype=str,
    encoding="utf-8-sig"
)

for nome, df in {
    "Avaliação": df_avaliacao,
    "Perfil": df_perfis
}.items():
    print(f"\n{nome}")
    print("Registros:", len(df))
    print("IDs ausentes:", df["ID"].isna().sum())
    print("IDs duplicados:", df["ID"].duplicated().sum())

ids_avaliacao = set(df_avaliacao["ID"].dropna())
ids_perfil = set(df_perfis["ID"].dropna())

print("\nParticipantes com avaliação e perfil:",
      len(ids_avaliacao & ids_perfil))

print("Somente avaliação:",
      len(ids_avaliacao - ids_perfil))

print("Somente perfil:",
      len(ids_perfil - ids_avaliacao))

print("Total de participantes distintos:",
      len(ids_avaliacao | ids_perfil))


Avaliação
Registros: 92
IDs ausentes: 0
IDs duplicados: 0

Perfil
Registros: 130
IDs ausentes: 0
IDs duplicados: 0

Participantes com avaliação e perfil: 80
Somente avaliação: 12
Somente perfil: 50
Total de participantes distintos: 142


In [2]:
def auditar_base(df, nome_base):
    registros = []

    for coluna in df.columns:
        serie = df[coluna]

        registros.append({
            "base": nome_base,
            "variavel": coluna,
            "total_registros": len(df),
            "preenchidos": serie.notna().sum(),
            "ausentes": serie.isna().sum(),
            "percentual_ausente": round(
                serie.isna().mean() * 100, 2
            ),
            "valores_distintos": serie.nunique(dropna=True),
            "tipo_dado": str(serie.dtype)
        })

    return pd.DataFrame(registros)


auditoria = pd.concat([
    auditar_base(df_avaliacao, "avaliacao"),
    auditar_base(df_perfis, "perfil")
], ignore_index=True)

auditoria = auditoria.sort_values(
    "percentual_ausente",
    ascending=False
)

display(auditoria)

auditoria.to_csv(
    PASTA_REPORTS / "auditoria_qualidade_inicial.csv",
    index=False,
    encoding="utf-8-sig"
)

,base,variavel,total_registros,preenchidos,ausentes,percentual_ausente,valores_distintos,tipo_dado
28,avaliacao,primeira_geracao_ensino_superior,92,0,92,100.00,0,str
29,avaliacao,primeira_geracao_pos_graduacao,92,0,92,100.00,0,str
53,perfil,primeira_geracao_pos_graduacao,130,0,130,100.00,0,str
22,avaliacao,distancia_deslocamento,92,1,91,98.91,1,str
15,avaliacao,genero,92,1,91,98.91,1,str
52,perfil,primeira_geracao_ensino_superior,130,13,117,90.00,2,str
57,perfil,acesso_estudantes_tecnologia,130,14,116,89.23,3,str
39,perfil,genero,130,14,116,89.23,2,str
54,perfil,area_docencia,130,14,116,89.23,5,str
55,perfil,experiencia_docente,130,14,116,89.23,3,str


In [3]:

colunas_avaliacao = [
    "ID", "grupo",
    "expectativas",
    "desistencia_projeto",
    "continuar_projeto",
    "representatividade",
    "motivacao_futuro",
    "ampliacao_oportunidades",
    "interesse_computacao_pos",
    "conhecimento_computacao_pos",
    "frequencia_computacao_pos",
    "perspectivas_percebidas_multirresposta",
    "etapas_interseccionalidade_docencia"
]

avaliacao = df_avaliacao[colunas_avaliacao].copy()


df_geral = avaliacao.merge(
    df_perfis,
    on="ID",
    how="outer",
    suffixes=("_avaliacao", "_perfil"),
    indicator=True,
    validate="one_to_one"
)


df_geral["situacao_registro"] = df_geral["_merge"].map({
    "both": "Avaliação e perfil",
    "left_only": "Somente avaliação",
    "right_only": "Somente perfil"
})


divergencias_grupo = df_geral.loc[
    df_geral["grupo_avaliacao"].notna()
    & df_geral["grupo_perfil"].notna()
    & (
        df_geral["grupo_avaliacao"]
        != df_geral["grupo_perfil"]
    )
]

print("Divergências de grupo:", len(divergencias_grupo))


if len(divergencias_grupo) == 0:
    df_geral["grupo"] = df_geral[
        "grupo_avaliacao"
    ].combine_first(df_geral["grupo_perfil"])

print("Dimensões:", df_geral.shape)
display(df_geral.head())

Divergências de grupo: 0
Dimensões: (142, 38)


,ID,grupo_avaliacao,expectativas,desistencia_projeto,continuar_projeto,representatividade,motivacao_futuro,ampliacao_oportunidades,interesse_computacao_pos,conhecimento_computacao_pos,...,semestre_graduacao_pos,primeira_geracao_ensino_superior,primeira_geracao_pos_graduacao,area_docencia,experiencia_docente,laboratorio_escola,acesso_estudantes_tecnologia,_merge,situacao_registro,grupo
0,PEB1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,right_only,Somente perfil,EB
1,PEB10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,right_only,Somente perfil,EB
2,PEB11,EB,5,Não,Sim,5.0,3.0,3.0,Moderado,Moderado,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,both,Avaliação e perfil,EB
3,PEB12,EB,5,Não,Sim,5.0,5.0,5.0,Muito,Moderado,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,both,Avaliação e perfil,EB
4,PEB13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,right_only,Somente perfil,EB


In [4]:
PASTA_INTERIM = PROJETO / "data" / "interim"
PASTA_INTERIM.mkdir(parents=True, exist_ok=True)

df_geral.to_csv(
    PASTA_INTERIM / "dashboard_base_geral_preliminar.csv",
    index=False,
    encoding="utf-8-sig"
)

In [5]:
df_perfis["beneficios_sociais_multirresposta"].value_counts(
    dropna=False
)

beneficios_sociais_multirresposta
Não                                             86
Bolsa Família                                   15
Bolsa Família, Pé de Meia                       13
Pé de Meia                                      10
Bolsa Família, Auxílio Brasil, Pé de Meia        1
Moeda Arariboia                                  1
Benefício de Prestação Continuada (BPC), Não     1
Auxílio Brasil                                   1
Não,                                             1
Loas                                             1
Name: count, dtype: int64

In [6]:
df_perfis["beneficios_sociais_multirresposta"].str.contains(
    "Bolsa Família",
    case=False,
    na=False
).sum()

np.int64(29)

In [7]:
df_perfis["beneficios_sociais_multirresposta"].str.contains(
    "Pé de Meia",
    case=False,
    na=False
).sum()

np.int64(24)

In [8]:
df_perfil_limpo = df_perfis.copy()

In [9]:
df_perfil_limpo["bolsa_familia"] = (
    df_perfil_limpo["beneficios_sociais_multirresposta"]
    .str.contains("Bolsa Família", case=False, na=False)
)

df_perfil_limpo["pe_de_meia"] = (
    df_perfil_limpo["beneficios_sociais_multirresposta"]
    .str.contains("Pé de Meia", case=False, na=False)
)

In [10]:
print(
    df_perfis["beneficios_sociais_multirresposta"]
    .dropna()
    .unique()
)

<StringArray>
[                   'Bolsa Família, Pé de Meia',
                                          'Não',
                                'Bolsa Família',
                                   'Pé de Meia',
    'Bolsa Família, Auxílio Brasil, Pé de Meia',
                              'Moeda Arariboia',
 'Benefício de Prestação Continuada (BPC), Não',
                               'Auxílio Brasil',
                                         'Não,',
                                         'Loas']
Length: 10, dtype: str


In [11]:
coluna = df_perfil_limpo["beneficios_sociais_multirresposta"]

df_perfil_limpo["auxilio_brasil"] = coluna.str.contains(
    "Auxílio Brasil", case=False, na=False
)

df_perfil_limpo["moeda_arariboia"] = coluna.str.contains(
    "Moeda Arariboia", case=False, na=False
)

df_perfil_limpo["bpc_loas"] = coluna.str.contains(
    r"BPC|Loas",
    case=False,
    na=False,
    regex=True
)

df_perfil_limpo["nenhum_beneficio"] = (
    coluna.str.strip().str.rstrip(",").str.lower() == "não"
)

df_perfil_limpo["beneficio_resposta_contraditoria"] = (
    coluna.str.contains(r"\bNão\b", case=False, na=False)
    & coluna.str.contains(
        r"BPC|Loas|Bolsa Família|Pé de Meia|Auxílio Brasil|Moeda Arariboia",
        case=False,
        na=False
    )
)

In [12]:
colunas_beneficios = [
    "bolsa_familia",
    "pe_de_meia",
    "auxilio_brasil",
    "moeda_arariboia",
    "bpc_loas",
    "nenhum_beneficio",
    "beneficio_resposta_contraditoria"
]

print(df_perfil_limpo[colunas_beneficios].sum())

bolsa_familia                       29
pe_de_meia                          24
auxilio_brasil                       2
moeda_arariboia                      1
bpc_loas                             2
nenhum_beneficio                    87
beneficio_resposta_contraditoria     1
dtype: int64


In [13]:
print(
    df_perfis["dispositivos_multirresposta"]
    .dropna()
    .unique()
)

<StringArray>
[                             'Smartphone/Celular',
         'Computador/Notebook, Smartphone/Celular',
                             'Computador/Notebook',
 'Computador/Notebook, Smartphone/Celular, Tablet',
                      'Smartphone/Celular, Tablet',
                                          'Tablet']
Length: 6, dtype: str


In [14]:
coluna = df_perfil_limpo["dispositivos_multirresposta"]

df_perfil_limpo["tem_celular"] = coluna.str.contains(
    "Smartphone/Celular", case=False, na=False, regex=False
)

df_perfil_limpo["tem_computador"] = coluna.str.contains(
    "Computador/Notebook", case=False, na=False, regex=False
)

df_perfil_limpo["tem_tablet"] = coluna.str.contains(
    "Tablet", case=False, na=False, regex=False
)

In [15]:
colunas_dispositivos = [
    "tem_celular",
    "tem_computador",
    "tem_tablet"
]

print(df_perfil_limpo[colunas_dispositivos].sum())

tem_celular       125
tem_computador     82
tem_tablet         24
dtype: int64


In [16]:
print(
    "Respostas ausentes:",
    df_perfil_limpo["dispositivos_multirresposta"].isna().sum()
)

Respostas ausentes: 0


In [17]:
print(
    df_perfis["internet_domestica_multirresposta"]
    .dropna()
    .unique()
)

<StringArray>
[                                                                                                       'Não tenho acesso à internet',
                                                                                               'Sim, banda larga (fibra óptica/cabo)',
                                       'Sim, mas apenas via rede móvel (dados do celular), Sim, mas com conexão limitada ou instável',
                                            'Sim, banda larga (fibra óptica/cabo), Sim, mas apenas via rede móvel (dados do celular)',
                                                                                          'Sim, mas com conexão limitada ou instável',
                                                                                  'Sim, mas apenas via rede móvel (dados do celular)',
                                                    'Sim, banda larga (fibra óptica/cabo), Sim, mas com conexão limitada ou instável',
 'Sim, banda larga (fibra óptica/cabo), S

In [18]:
coluna = df_perfil_limpo["internet_domestica_multirresposta"]

df_perfil_limpo["internet_sem_acesso"] = coluna.str.contains(
    "Não tenho acesso à internet",
    case=False, na=False, regex=False
)

df_perfil_limpo["internet_banda_larga"] = coluna.str.contains(
    "Sim, banda larga",
    case=False, na=False, regex=False
)

df_perfil_limpo["internet_rede_movel"] = coluna.str.contains(
    "Sim, mas apenas via rede móvel",
    case=False, na=False, regex=False
)

df_perfil_limpo["internet_limitada"] = coluna.str.contains(
    "Sim, mas com conexão limitada ou instável",
    case=False, na=False, regex=False
)

In [19]:
colunas_internet = [
    "internet_sem_acesso",
    "internet_banda_larga",
    "internet_rede_movel",
    "internet_limitada"
]

print(df_perfil_limpo[colunas_internet].sum())

internet_sem_acesso       1
internet_banda_larga    121
internet_rede_movel      21
internet_limitada         7
dtype: int64


In [20]:
print(
    "Respostas ausentes:",
    coluna.isna().sum()
)

contradicoes = df_perfil_limpo[
    df_perfil_limpo["internet_sem_acesso"]
    & (
        df_perfil_limpo["internet_banda_larga"]
        | df_perfil_limpo["internet_rede_movel"]
        | df_perfil_limpo["internet_limitada"]
    )
]

print("Respostas contraditórias:", len(contradicoes))

Respostas ausentes: 0
Respostas contraditórias: 0


In [21]:
print(df_perfil_limpo.columns.tolist())

['ID', 'grupo', 'idade', 'raca_etnia', 'genero', 'cidade_uf', 'renda_familiar_faixa_sm', 'beneficios_sociais_multirresposta', 'dispositivos_multirresposta', 'internet_domestica_multirresposta', 'espaco_estudo', 'distancia_deslocamento', 'responsabilidades_familiares_original', 'instituicao_vinculo', 'ano_escolar_eb', 'turno_estudo_eb', 'semestre_graduacao_pos', 'primeira_geracao_ensino_superior', 'primeira_geracao_pos_graduacao', 'area_docencia', 'experiencia_docente', 'laboratorio_escola', 'acesso_estudantes_tecnologia', 'bolsa_familia', 'pe_de_meia', 'auxilio_brasil', 'moeda_arariboia', 'bpc_loas', 'nenhum_beneficio', 'beneficio_resposta_contraditoria', 'tem_celular', 'tem_computador', 'tem_tablet', 'internet_sem_acesso', 'internet_banda_larga', 'internet_rede_movel', 'internet_limitada']


In [22]:
print(
    df_perfil_limpo["idade"]
    .value_counts(dropna=False)
    .sort_index()
)

idade
12.0     4
13.0     7
14.0    13
15.0    13
16.0    27
17.0     8
18.0     4
19.0     8
20.0     5
21.0    11
22.0     2
23.0     1
24.0     2
25.0     1
26.0     1
27.0     3
29.0     3
30.0     3
31.0     1
37.0     1
41.0     1
42.0     1
44.0     1
45.0     2
46.0     2
58.0     1
60.0     1
70.0     1
NaN      2
Name: count, dtype: int64


In [23]:
df_perfil_limpo["idade_num"] = pd.to_numeric(
    df_perfil_limpo["idade"],
    errors="coerce"
)

print("Idades ausentes ou inválidas:",
      df_perfil_limpo["idade_num"].isna().sum())

print("Menor idade:", df_perfil_limpo["idade_num"].min())
print("Maior idade:", df_perfil_limpo["idade_num"].max())

Idades ausentes ou inválidas: 2
Menor idade: 12.0
Maior idade: 70.0


In [24]:
display(
    df_perfil_limpo.groupby("grupo")["idade_num"]
    .agg(["count", "min", "max", "median"])
)

,count,min,max,median
grupo,,,,
EB,74,12.0,18.0,16.0
GRAD,36,14.0,30.0,21.0
POS,5,24.0,30.0,29.0
PROF_EB,13,29.0,70.0,45.0


In [25]:
df_perfil_limpo["faixa_etaria"] = pd.cut(
    df_perfil_limpo["idade_num"],
    bins=[0, 17, 24, 34, 44, 59, float("inf")],
    labels=[
        "12 a 17 anos",
        "18 a 24 anos",
        "25 a 34 anos",
        "35 a 44 anos",
        "45 a 59 anos",
        "60 anos ou mais"
    ]
)

In [26]:
print(
    df_perfil_limpo["faixa_etaria"]
    .value_counts(dropna=False)
    .sort_index()
)

faixa_etaria
12 a 17 anos       72
18 a 24 anos       33
25 a 34 anos       12
35 a 44 anos        4
45 a 59 anos        5
60 anos ou mais     2
NaN                 2
Name: count, dtype: int64


In [27]:
print(
    "Idades válidas:",
    df_perfil_limpo["idade_num"].notna().sum()
)

print(
    "Faixas preenchidas:",
    df_perfil_limpo["faixa_etaria"].notna().sum()
)

Idades válidas: 128
Faixas preenchidas: 128


In [28]:
df_perfil_limpo["raca_etnia"].value_counts(dropna=False)

raca_etnia
Branca      55
Parda       48
Preta       26
Indígena     1
Name: count, dtype: int64

In [29]:
df_perfil_limpo["genero"].value_counts(dropna=False)

genero
NaN                   116
Mulher cisgênero       13
Mulher transgênero      1
Name: count, dtype: int64

In [30]:
pd.crosstab(
    df_perfil_limpo["grupo"],
    df_perfil_limpo["genero"].fillna("Não informado")
)

genero,Mulher cisgênero,Mulher transgênero,Não informado
grupo,,,
EB,0,0,74
GRAD,12,1,24
POS,0,0,5
PROF_EB,1,0,13


In [31]:
df_perfil_limpo["genero_informado"] = (
    df_perfil_limpo["genero"].notna()
)

In [32]:
df_perfil_limpo["renda_familiar_faixa_sm"].value_counts(
    dropna=False
)

renda_familiar_faixa_sm
De 1 a 3 salários mínimos      52
Até 1 salário mínimo           42
Acima de 5 salários mínimos    20
De 3 a 5 salários mínimos      16
Name: count, dtype: int64

In [33]:
ordem_renda = [
    "Até 1 salário mínimo",
    "De 1 a 3 salários mínimos",
    "De 3 a 5 salários mínimos",
    "Acima de 5 salários mínimos"
]

df_perfil_limpo["renda_familiar_faixa_sm"] = pd.Categorical(
    df_perfil_limpo["renda_familiar_faixa_sm"],
    categories=ordem_renda,
    ordered=True
)

print(
    df_perfil_limpo["renda_familiar_faixa_sm"]
    .value_counts()
    .sort_index()
)

renda_familiar_faixa_sm
Até 1 salário mínimo           42
De 1 a 3 salários mínimos      52
De 3 a 5 salários mínimos      16
Acima de 5 salários mínimos    20
Name: count, dtype: int64


In [34]:
df_perfil_limpo["espaco_estudo"].value_counts(dropna=False)

espaco_estudo
Sim, tenho um espaço tranquilo e dedicado                    66
Sim, mas compartilho com outras pessoas                      34
Não, tenho dificuldades para encontrar um espaço adequado    15
NaN                                                          14
Não tenho um lugar específico                                 1
Name: count, dtype: int64

In [35]:
mapeamento_espaco = {
    "Sim, tenho um espaço tranquilo e dedicado": "Espaço dedicado",
    "Sim, mas compartilho com outras pessoas": "Espaço compartilhado",
    "Não, tenho dificuldades para encontrar um espaço adequado": "Sem espaço adequado",
    "Não tenho um lugar específico": "Sem espaço adequado"
}

df_perfil_limpo["espaco_estudo_categoria"] = (
    df_perfil_limpo["espaco_estudo"]
    .map(mapeamento_espaco)
)

In [36]:
df_perfil_limpo["espaco_estudo_categoria"].value_counts(
    dropna=False
)

espaco_estudo_categoria
Espaço dedicado         66
Espaço compartilhado    34
Sem espaço adequado     16
NaN                     14
Name: count, dtype: int64

In [37]:
df_perfil_limpo["distancia_deslocamento"].value_counts(
    dropna=False
)

distancia_deslocamento
NaN                                            98
Longe (mais de 30 minutos até 1 hora)          12
Perto (até 15 minutos)                          6
Muito perto (a poucos minutos)                  5
Muito perto (vou andando em poucos minutos)     5
Um pouco longe (entre 15 e 30 minutos)          4
Name: count, dtype: int64

In [38]:
mapeamento_distancia = {
    "Muito perto (a poucos minutos)": "Muito perto",
    "Muito perto (vou andando em poucos minutos)": "Muito perto",
    "Perto (até 15 minutos)": "Perto",
    "Um pouco longe (entre 15 e 30 minutos)": "Um pouco longe",
    "Longe (mais de 30 minutos até 1 hora)": "Longe"
}

df_perfil_limpo["distancia_categoria"] = (
    df_perfil_limpo["distancia_deslocamento"]
    .map(mapeamento_distancia)
)

In [39]:
df_perfil_limpo["distancia_categoria"].value_counts(
    dropna=False
)

distancia_categoria
NaN               98
Longe             12
Muito perto       10
Perto              6
Um pouco longe     4
Name: count, dtype: int64

In [40]:
pd.crosstab(
    df_perfil_limpo["grupo"],
    df_perfil_limpo["distancia_categoria"].fillna("Não informado")
)

distancia_categoria,Longe,Muito perto,Não informado,Perto,Um pouco longe
grupo,,,,,
EB,6,5,55,4,4
GRAD,6,5,24,2,0
POS,0,0,5,0,0
PROF_EB,0,0,14,0,0


In [41]:
df_perfil_limpo["responsabilidades_familiares_original"].value_counts(
    dropna=False
)

responsabilidades_familiares_original
Não                                        102
Não, mas ajudo com meus irmãos menores.     15
Sim, mas não sou mãe solo                    8
Sim, sou mãe solo                            3
Não tenho filhos                             2
Name: count, dtype: int64

In [43]:
coluna = "responsabilidades_familiares_original"

print("Responderam:", df_perfil_limpo[coluna].notna().sum())
print("Não responderam:", df_perfil_limpo[coluna].isna().sum())

Responderam: 130
Não responderam: 0


In [44]:
pd.crosstab(
    df_perfil_limpo["grupo"],
    df_perfil_limpo[coluna].notna(),
    rownames=["Grupo"],
    colnames=["Respondeu"]
)

Respondeu,True
Grupo,
EB,74
GRAD,37
POS,5
PROF_EB,14


In [45]:
df_perfil_limpo["instituicao_vinculo"].value_counts(
    dropna=False
)

instituicao_vinculo
Colégio Estadual Paulo Assis Ribeiro (CEPAR)                                                           11
Escola Básica Municipal Professora Herondina Medeiros Zeferino                                          9
Colégio Estadual Governador Flávio Marcílio                                                             8
Instituto Federal de Educação, Ciência e Tecnologia Goiano (IF Goiano) - Campus Ceres                   8
Escola Estadual de Ensino Fundamental Professora Antonia Paes da Silva                                  8
UFPA - Universidade Federal do Pará                                                                     8
Escola Estadual Ângelo Ramazzotti                                                                       7
Escola Municipal Índio Piragibe                                                                         7
Instituto Federal de Educação, Ciência e Tecnologia de Santa Catarina (IFSC) - Campus Florianópolis     7
UFC - Universidade Federal

In [46]:
# Criar uma cópia da coluna original
df_perfil_limpo["instituicao_padronizada"] = (
    df_perfil_limpo["instituicao_vinculo"].copy()
)

# Dicionário de padronização
mapeamento_instituicoes = {
    "Escola municipal altivo cesar": "Escola Municipal Altivo César",
    "E.M Altivo César": "Escola Municipal Altivo César",
    "EBM Herondina Medeiros Zeferino": "Escola Básica Municipal Professora Herondina Medeiros Zeferino"
}

# Aplicar as substituições
df_perfil_limpo["instituicao_padronizada"] = (
    df_perfil_limpo["instituicao_padronizada"]
    .replace(mapeamento_instituicoes)
)

In [47]:
print("Instituições antes:",
      df_perfil_limpo["instituicao_vinculo"].nunique())

print("Instituições depois:",
      df_perfil_limpo["instituicao_padronizada"].nunique())

display(
    df_perfil_limpo["instituicao_padronizada"]
    .value_counts(dropna=False)
)

Instituições antes: 26
Instituições depois: 23


instituicao_padronizada
Colégio Estadual Paulo Assis Ribeiro (CEPAR)                                                           11
Escola Básica Municipal Professora Herondina Medeiros Zeferino                                         10
Colégio Estadual Governador Flávio Marcílio                                                             8
Instituto Federal de Educação, Ciência e Tecnologia Goiano (IF Goiano) - Campus Ceres                   8
Escola Estadual de Ensino Fundamental Professora Antonia Paes da Silva                                  8
UFPA - Universidade Federal do Pará                                                                     8
Escola Estadual Ângelo Ramazzotti                                                                       7
Escola Municipal Índio Piragibe                                                                         7
Instituto Federal de Educação, Ciência e Tecnologia de Santa Catarina (IFSC) - Campus Florianópolis     7
UFC - Universidade Fed

In [48]:
df_instituicoes = pd.DataFrame({
    "instituicao_padronizada": (
        df_perfil_limpo["instituicao_padronizada"]
        .dropna()
        .unique()
    )
})

df_instituicoes = df_instituicoes.sort_values(
    "instituicao_padronizada"
).reset_index(drop=True)

df_instituicoes["estado_instituicao"] = pd.NA
df_instituicoes["regiao_instituicao"] = pd.NA

display(df_instituicoes)

,instituicao_padronizada,estado_instituicao,regiao_instituicao
0,Altivo César,<NA>,<NA>
1,Centro Educacional 310 de Santa Maria,<NA>,<NA>
2,Colégio Estadual Governador Flávio Marcílio,<NA>,<NA>
3,Colégio Estadual Paulo Assis Ribeiro (CEPAR),<NA>,<NA>
4,Escola Básica Municipal Professora Herondina M...,<NA>,<NA>
5,Escola Estadual Delfim Moreira,<NA>,<NA>
6,Escola Estadual Delfim Moreira (Central),<NA>,<NA>
7,Escola Estadual Duque de Caxias,<NA>,<NA>
8,Escola Estadual de Ensino Fundamental Professo...,<NA>,<NA>
9,Escola Estadual Ângelo Ramazzotti,<NA>,<NA>


In [49]:
estados_conhecidos = {
    "UFC - Universidade Federal do Ceará": "CE",
    "UFPA - Universidade Federal do Pará": "PA",
    "UFPB - Universidade Federal da Paraíba": "PB",
    "UFJF - Universidade Federal de Juiz de Fora": "MG",
    "UFF - Universidade Federal Fluminense": "RJ",
    "UFSC - Universidade Federal de Santa Catarina": "SC",
    "UnB - Universidade de Brasília": "DF",
    "UFAM - Universidade Federal do Amazonas": "AM",
    "IF Goiano - Instituto Federal de Educação, Ciência e Tecnologia Goiano": "GO",
    "Instituto Federal de Educação, Ciência e Tecnologia Goiano (IF Goiano) - Campus Ceres": "GO",
    "Instituto Federal de Educação, Ciência e Tecnologia de Santa Catarina (IFSC) - Campus Florianópolis": "SC"
}

df_instituicoes["estado_instituicao"] = (
    df_instituicoes["instituicao_padronizada"]
    .map(estados_conhecidos)
)

In [50]:
regioes = {
    "AC": "Norte", "AP": "Norte", "AM": "Norte",
    "PA": "Norte", "RO": "Norte", "RR": "Norte",
    "TO": "Norte",

    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste",
    "MA": "Nordeste", "PB": "Nordeste", "PE": "Nordeste",
    "PI": "Nordeste", "RN": "Nordeste", "SE": "Nordeste",

    "DF": "Centro-Oeste", "GO": "Centro-Oeste",
    "MT": "Centro-Oeste", "MS": "Centro-Oeste",

    "ES": "Sudeste", "MG": "Sudeste",
    "RJ": "Sudeste", "SP": "Sudeste",

    "PR": "Sul", "RS": "Sul", "SC": "Sul"
}

df_instituicoes["regiao_instituicao"] = (
    df_instituicoes["estado_instituicao"]
    .map(regioes)
)

In [51]:
display(
    df_instituicoes.loc[
        df_instituicoes["estado_instituicao"].isna(),
        ["instituicao_padronizada"]
    ]
)

,instituicao_padronizada
0,Altivo César
1,Centro Educacional 310 de Santa Maria
2,Colégio Estadual Governador Flávio Marcílio
3,Colégio Estadual Paulo Assis Ribeiro (CEPAR)
4,Escola Básica Municipal Professora Herondina M...
5,Escola Estadual Delfim Moreira
6,Escola Estadual Delfim Moreira (Central)
7,Escola Estadual Duque de Caxias
8,Escola Estadual de Ensino Fundamental Professo...
9,Escola Estadual Ângelo Ramazzotti


In [52]:
display(
    df_instituicoes[
        [
            "instituicao_padronizada",
            "estado_instituicao",
            "regiao_instituicao"
        ]
    ]
)

,instituicao_padronizada,estado_instituicao,regiao_instituicao
0,Altivo César,NaN,NaN
1,Centro Educacional 310 de Santa Maria,NaN,NaN
2,Colégio Estadual Governador Flávio Marcílio,NaN,NaN
3,Colégio Estadual Paulo Assis Ribeiro (CEPAR),NaN,NaN
4,Escola Básica Municipal Professora Herondina M...,NaN,NaN
5,Escola Estadual Delfim Moreira,NaN,NaN
6,Escola Estadual Delfim Moreira (Central),NaN,NaN
7,Escola Estadual Duque de Caxias,NaN,NaN
8,Escola Estadual de Ensino Fundamental Professo...,NaN,NaN
9,Escola Estadual Ângelo Ramazzotti,NaN,NaN


In [54]:
df_perfil_limpo["instituicao_padronizada"] = (
    df_perfil_limpo["instituicao_padronizada"].replace({
        "Instituto Federal de Educação, Ciência e Tecnologia Goiano (IF Goiano) - Campus Ceres":
        "IF Goiano - Instituto Federal de Educação, Ciência e Tecnologia Goiano - Campus Ceres"
    })
)

In [55]:
df_instituicoes["instituicao_padronizada"] = (
    df_instituicoes["instituicao_padronizada"].replace({
        "Instituto Federal de Educação, Ciência e Tecnologia Goiano (IF Goiano) - Campus Ceres":
        "IF Goiano - Instituto Federal de Educação, Ciência e Tecnologia Goiano - Campus Ceres"
    })
)

In [56]:
display(
    df_instituicoes[
        df_instituicoes["instituicao_padronizada"]
        .str.contains("IF Goiano", na=False)
    ]
)

,instituicao_padronizada,estado_instituicao,regiao_instituicao
12,"IF Goiano - Instituto Federal de Educação, Ciê...",GO,Centro-Oeste
13,"IF Goiano - Instituto Federal de Educação, Ciê...",GO,Centro-Oeste


In [57]:
nome_if_goiano = (
    "IF Goiano - Instituto Federal de Educação, "
    "Ciência e Tecnologia Goiano"
)

df_perfil_limpo["instituicao_padronizada"] = (
    df_perfil_limpo["instituicao_padronizada"].replace({
        nome_if_goiano + " - Campus Ceres": nome_if_goiano
    })
)

In [58]:
df_instituicoes["instituicao_padronizada"] = (
    df_instituicoes["instituicao_padronizada"].replace({
        nome_if_goiano + " - Campus Ceres": nome_if_goiano
    })
)

df_instituicoes = df_instituicoes.drop_duplicates(
    subset=["instituicao_padronizada"]
).reset_index(drop=True)

In [59]:
display(
    df_perfil_limpo[
        df_perfil_limpo["instituicao_padronizada"]
        .str.contains("IF Goiano", na=False)
    ][
        ["instituicao_vinculo", "instituicao_padronizada"]
    ]
)

print(
    "Total de instituições padronizadas:",
    df_instituicoes["instituicao_padronizada"].nunique()
)

,instituicao_vinculo,instituicao_padronizada
2,"Instituto Federal de Educação, Ciência e Tecno...","IF Goiano - Instituto Federal de Educação, Ciê..."
7,"Instituto Federal de Educação, Ciência e Tecno...","IF Goiano - Instituto Federal de Educação, Ciê..."
12,"Instituto Federal de Educação, Ciência e Tecno...","IF Goiano - Instituto Federal de Educação, Ciê..."
47,"Instituto Federal de Educação, Ciência e Tecno...","IF Goiano - Instituto Federal de Educação, Ciê..."
48,"Instituto Federal de Educação, Ciência e Tecno...","IF Goiano - Instituto Federal de Educação, Ciê..."
52,"Instituto Federal de Educação, Ciência e Tecno...","IF Goiano - Instituto Federal de Educação, Ciê..."
62,"Instituto Federal de Educação, Ciência e Tecno...","IF Goiano - Instituto Federal de Educação, Ciê..."
84,"IF Goiano - Instituto Federal de Educação, Ciê...","IF Goiano - Instituto Federal de Educação, Ciê..."
93,"IF Goiano - Instituto Federal de Educação, Ciê...","IF Goiano - Instituto Federal de Educação, Ciê..."
103,"IF Goiano - Instituto Federal de Educação, Ciê...","IF Goiano - Instituto Federal de Educação, Ciê..."


Total de instituições padronizadas: 22


In [60]:
mapeamento_escolas = {
    "Altivo César": "Escola Municipal Altivo César",
    "Escola Estadual Delfim Moreira (Central)": "Escola Estadual Delfim Moreira"
}

# Atualizar a base de perfil
df_perfil_limpo["instituicao_padronizada"] = (
    df_perfil_limpo["instituicao_padronizada"]
    .replace(mapeamento_escolas)
)

# Atualizar a tabela de instituições
df_instituicoes["instituicao_padronizada"] = (
    df_instituicoes["instituicao_padronizada"]
    .replace(mapeamento_escolas)
)

df_instituicoes = df_instituicoes.drop_duplicates(
    subset="instituicao_padronizada"
).reset_index(drop=True)

In [61]:
estados_escolas = {
    "Escola Municipal Altivo César": "RJ",
    "Centro Educacional 310 de Santa Maria": "DF",
    "Colégio Estadual Governador Flávio Marcílio": "CE",
    "Colégio Estadual Paulo Assis Ribeiro (CEPAR)": "RJ",
    "Escola Básica Municipal Professora Herondina Medeiros Zeferino": "SC",
    "Escola Estadual Delfim Moreira": "MG",
    "Escola Estadual de Ensino Fundamental Professora Antonia Paes da Silva": "PA",
    "Escola Estadual Ângelo Ramazzotti": "AM",
    "Escola Municipal Índio Piragibe": "PB"
}

# Preencher o estado das escolas identificadas
df_instituicoes["estado_instituicao"] = (
    df_instituicoes["instituicao_padronizada"]
    .map(estados_escolas)
    .combine_first(df_instituicoes["estado_instituicao"])
)

# Atualizar as regiões usando o dicionário já criado
df_instituicoes["regiao_instituicao"] = (
    df_instituicoes["estado_instituicao"]
    .map(regioes)
)

In [62]:
display(
    df_instituicoes.loc[
        df_instituicoes["estado_instituicao"].isna(),
        ["instituicao_padronizada"]
    ]
)

print(
    "Total de instituições:",
    df_instituicoes["instituicao_padronizada"].nunique()
)

,instituicao_padronizada
6,Escola Estadual Duque de Caxias


Total de instituições: 20


In [63]:
df_instituicoes.loc[
    df_instituicoes["instituicao_padronizada"]
    == "Escola Estadual Duque de Caxias",
    ["estado_instituicao", "regiao_instituicao"]
] = pd.NA

In [64]:
df_perfil_limpo = df_perfil_limpo.merge(
    df_instituicoes[
        [
            "instituicao_padronizada",
            "estado_instituicao",
            "regiao_instituicao"
        ]
    ],
    on="instituicao_padronizada",
    how="left",
    validate="many_to_one"
)

In [65]:
display(
    df_perfil_limpo[
        [
            "instituicao_padronizada",
            "estado_instituicao",
            "regiao_instituicao"
        ]
    ].head(15)
)

print("Total de participantes:", len(df_perfil_limpo))

print(
    "Participantes sem estado identificado:",
    df_perfil_limpo["estado_instituicao"].isna().sum()
)

,instituicao_padronizada,estado_instituicao,regiao_instituicao
0,Colégio Estadual Governador Flávio Marcílio,CE,Nordeste
1,Escola Básica Municipal Professora Herondina M...,SC,Sul
2,"IF Goiano - Instituto Federal de Educação, Ciê...",GO,Centro-Oeste
3,Escola Estadual Ângelo Ramazzotti,AM,Norte
4,Escola Municipal Altivo César,RJ,Sudeste
5,Colégio Estadual Paulo Assis Ribeiro (CEPAR),RJ,Sudeste
6,Centro Educacional 310 de Santa Maria,DF,Centro-Oeste
7,"IF Goiano - Instituto Federal de Educação, Ciê...",GO,Centro-Oeste
8,Colégio Estadual Governador Flávio Marcílio,CE,Nordeste
9,Escola Estadual de Ensino Fundamental Professo...,PA,Norte


Total de participantes: 130
Participantes sem estado identificado: 5


In [66]:
df_perfil_limpo["ano_escolar_eb"].value_counts(
    dropna=False
)

ano_escolar_eb
NaN                                                 56
2° ano - Ensino Médio                               26
8° ano - Ensino Fundamental                         12
9° ano - Ensino Fundamental                         11
3° ano - Ensino Médio                               11
1° ano - Ensino Médio                                9
7⁰ ano                                               1
7° ano                                               1
4º ano do ensino médio integrado                     1
7° ano - Ensino Fundamental                          1
3 fase de Eletrotécnica - ensino medio integrado     1
Name: count, dtype: int64

In [67]:
pd.crosstab(
    df_perfil_limpo["grupo"],
    df_perfil_limpo["ano_escolar_eb"].fillna("Não informado")
)

ano_escolar_eb,1° ano - Ensino Médio,2° ano - Ensino Médio,3 fase de Eletrotécnica - ensino medio integrado,3° ano - Ensino Médio,4º ano do ensino médio integrado,7° ano,7° ano - Ensino Fundamental,7⁰ ano,8° ano - Ensino Fundamental,9° ano - Ensino Fundamental,Não informado
grupo,,,,,,,,,,,
EB,9,26,1,11,1,1,1,1,12,11,0
GRAD,0,0,0,0,0,0,0,0,0,0,37
POS,0,0,0,0,0,0,0,0,0,0,5
PROF_EB,0,0,0,0,0,0,0,0,0,0,14


In [68]:
mapeamento_ano_escolar = {
    "7º ano": "7º ano - Ensino Fundamental",
    "7º ano - Ensino Fundamental": "7º ano - Ensino Fundamental",
    "8º ano - Ensino Fundamental": "8º ano - Ensino Fundamental",
    "9º ano - Ensino Fundamental": "9º ano - Ensino Fundamental",
    "1º ano - Ensino Médio": "1º ano - Ensino Médio",
    "2º ano - Ensino Médio": "2º ano - Ensino Médio",
    "3º ano - Ensino Médio": "3º ano - Ensino Médio",
    "4º ano do ensino médio integrado": "Ensino Médio Integrado - 4º ano",
    "3 fase de Eletrotécnica - ensino medio integrado":
        "Ensino Médio Integrado - 3ª fase (Eletrotécnica)"
}

df_perfil_limpo["ano_escolar_padronizado"] = (
    df_perfil_limpo["ano_escolar_eb"]
    .replace(mapeamento_ano_escolar)
)

In [69]:
print(
    df_perfil_limpo["ano_escolar_padronizado"]
    .value_counts(dropna=False)
)

ano_escolar_padronizado
NaN                                                 56
2° ano - Ensino Médio                               26
8° ano - Ensino Fundamental                         12
9° ano - Ensino Fundamental                         11
3° ano - Ensino Médio                               11
1° ano - Ensino Médio                                9
7⁰ ano                                               1
7° ano                                               1
Ensino Médio Integrado - 4º ano                      1
7° ano - Ensino Fundamental                          1
Ensino Médio Integrado - 3ª fase (Eletrotécnica)     1
Name: count, dtype: int64


In [70]:
pd.crosstab(
    df_perfil_limpo["grupo"],
    df_perfil_limpo["ano_escolar_padronizado"].notna()
)

ano_escolar_padronizado,False,True
grupo,,
EB,0,74
GRAD,37,0
POS,5,0
PROF_EB,14,0


In [71]:
df_perfil_limpo["ano_escolar_padronizado"] = (
    df_perfil_limpo["ano_escolar_padronizado"]
    .replace({
        "7⁰ ano": "7º ano - Ensino Fundamental",
        "7° ano": "7º ano - Ensino Fundamental",
        "7° ano - Ensino Fundamental": "7º ano - Ensino Fundamental"
    })
)

In [72]:
df_perfil_limpo["ano_escolar_padronizado"] = (
    df_perfil_limpo["ano_escolar_padronizado"]
    .str.replace("°", "º", regex=False)
)

In [73]:
df_perfil_limpo["ano_escolar_padronizado"].value_counts(
    dropna=False
)

ano_escolar_padronizado
NaN                                                 56
2º ano - Ensino Médio                               26
8º ano - Ensino Fundamental                         12
9º ano - Ensino Fundamental                         11
3º ano - Ensino Médio                               11
1º ano - Ensino Médio                                9
7º ano - Ensino Fundamental                          3
Ensino Médio Integrado - 4º ano                      1
Ensino Médio Integrado - 3ª fase (Eletrotécnica)     1
Name: count, dtype: int64

In [74]:
pd.crosstab(
    df_perfil_limpo["grupo"],
    df_perfil_limpo["ano_escolar_padronizado"].notna()
)

ano_escolar_padronizado,False,True
grupo,,
EB,0,74
GRAD,37,0
POS,5,0
PROF_EB,14,0


In [75]:
df_perfil_limpo["turno_estudo_eb"].value_counts(
    dropna=False
)

turno_estudo_eb
NaN         56
Manhã       37
Integral    18
Tarde       18
integral     1
Name: count, dtype: int64

In [76]:
pd.crosstab(
    df_perfil_limpo["grupo"],
    df_perfil_limpo["turno_estudo_eb"].fillna("Não informado")
)

turno_estudo_eb,Integral,Manhã,Não informado,Tarde,integral
grupo,,,,,
EB,18,37,0,18,1
GRAD,0,0,37,0,0
POS,0,0,5,0,0
PROF_EB,0,0,14,0,0


In [77]:
df_perfil_limpo["turno_estudo_padronizado"] = (
    df_perfil_limpo["turno_estudo_eb"]
    .replace({
        "integral": "Integral"
    })
)

In [78]:
df_perfil_limpo["turno_estudo_padronizado"].value_counts(
    dropna=False
)

turno_estudo_padronizado
NaN         56
Manhã       37
Integral    19
Tarde       18
Name: count, dtype: int64

In [79]:
df_perfil_limpo["semestre_graduacao_pos"].value_counts(
    dropna=False
)

semestre_graduacao_pos
NaN                                      88
3°                                        9
5°                                        7
2°                                        7
8°                                        5
7°                                        4
6°                                        4
4°                                        3
1°                                        2
Estou terminando o ensino fundamental     1
Name: count, dtype: int64

In [80]:
pd.crosstab(
    df_perfil_limpo["grupo"],
    df_perfil_limpo["semestre_graduacao_pos"].fillna("Não informado")
)

semestre_graduacao_pos,1°,2°,3°,4°,5°,6°,7°,8°,Estou terminando o ensino fundamental,Não informado
grupo,,,,,,,,,,
EB,0,0,0,0,0,0,0,0,0,74
GRAD,1,6,8,2,6,4,4,5,1,0
POS,1,1,1,1,1,0,0,0,0,0
PROF_EB,0,0,0,0,0,0,0,0,0,14


In [81]:
df_perfil_limpo.loc[
    df_perfil_limpo["semestre_graduacao_pos"]
    .str.contains("ensino fundamental", case=False, na=False),
    ["grupo", "semestre_graduacao_pos"]
]

,grupo,semestre_graduacao_pos
81,GRAD,Estou terminando o ensino fundamental


In [82]:
pd.crosstab(
    df_perfil_limpo["grupo"],
    df_perfil_limpo["semestre_graduacao_pos"].notna(),
    rownames=["Grupo"],
    colnames=["Possui informação de semestre"]
)

Possui informação de semestre,False,True
Grupo,,
EB,74,0
GRAD,0,37
POS,0,5
PROF_EB,14,0


In [83]:
df_perfil_limpo["semestre_num"] = pd.to_numeric(
    df_perfil_limpo["semestre_graduacao_pos"]
    .str.extract(r"(\d+)")[0],
    errors="coerce"
)

In [84]:
df_perfil_limpo["semestre_inconsistente"] = (
    df_perfil_limpo["semestre_graduacao_pos"].notna()
    & df_perfil_limpo["semestre_num"].isna()
)

In [85]:
print(
    df_perfil_limpo["semestre_num"]
    .value_counts(dropna=False)
    .sort_index()
)

print(
    "Respostas inconsistentes:",
    df_perfil_limpo["semestre_inconsistente"].sum()
)

semestre_num
1.0     2
2.0     7
3.0     9
4.0     3
5.0     7
6.0     4
7.0     4
8.0     5
NaN    89
Name: count, dtype: int64
Respostas inconsistentes: 1


In [86]:
df_perfil_limpo["primeira_geracao_ensino_superior"].value_counts(
    dropna=False
)

primeira_geracao_ensino_superior
NaN    117
Não     11
Sim      2
Name: count, dtype: int64

In [87]:
pd.crosstab(
    df_perfil_limpo["grupo"],
    df_perfil_limpo["primeira_geracao_ensino_superior"].fillna("Não informado")
)

primeira_geracao_ensino_superior,Não,Não informado,Sim
grupo,,,
EB,0,74,0
GRAD,11,24,2
POS,0,5,0
PROF_EB,0,14,0


In [88]:
df_perfil_limpo["primeira_geracao_pos_graduacao"].value_counts(
    dropna=False
)

primeira_geracao_pos_graduacao
NaN    130
Name: count, dtype: int64

In [89]:
pd.crosstab(
    df_perfil_limpo["grupo"],
    df_perfil_limpo["primeira_geracao_pos_graduacao"].fillna("Não informado")
)

primeira_geracao_pos_graduacao,Não informado
grupo,
EB,74
GRAD,37
POS,5
PROF_EB,14


In [90]:
df_perfil_limpo["area_docencia"].value_counts(
    dropna=False
)

area_docencia
NaN                                                              116
Ciências da Natureza (Física, Química, Biologia)                   5
Linguagens (Português, Inglês, Artes, Educação Física)             4
Matemática                                                         3
Tecnologias e Computação                                           1
Ciências Humanas (História, Geografia, Sociologia, Filosofia)      1
Name: count, dtype: int64

In [91]:
pd.crosstab(
    df_perfil_limpo["grupo"],
    df_perfil_limpo["area_docencia"].fillna("Não informado")
)

area_docencia,"Ciências Humanas (História, Geografia, Sociologia, Filosofia)","Ciências da Natureza (Física, Química, Biologia)","Linguagens (Português, Inglês, Artes, Educação Física)",Matemática,Não informado,Tecnologias e Computação
grupo,,,,,,
EB,0,0,0,0,74,0
GRAD,0,0,0,0,37,0
POS,0,0,0,0,5,0
PROF_EB,1,5,4,3,0,1


In [92]:
df_perfil_limpo["experiencia_docente"].value_counts(
    dropna=False
)

experiencia_docente
NaN                  116
Mais de 10 anos       12
Entre 6 e 10 anos      1
Entre 1 e 5 anos       1
Name: count, dtype: int64

In [93]:
pd.crosstab(
    df_perfil_limpo["grupo"],
    df_perfil_limpo["experiencia_docente"].fillna("Não informado")
)

experiencia_docente,Entre 1 e 5 anos,Entre 6 e 10 anos,Mais de 10 anos,Não informado
grupo,,,,
EB,0,0,0,74
GRAD,0,0,0,37
POS,0,0,0,5
PROF_EB,1,1,12,0


In [94]:
df_perfil_limpo["laboratorio_escola"].value_counts(
    dropna=False
)

laboratorio_escola
NaN                                                             116
Sim, há laboratório de informática bem equipado                   7
Não há laboratório, mas há alguns computadores disponíveis        3
Sim, mas os equipamentos são insuficientes ou desatualizados      2
Não há computadores ou recursos tecnológicos disponíveis          2
Name: count, dtype: int64

In [95]:
pd.crosstab(
    df_perfil_limpo["grupo"],
    df_perfil_limpo["laboratorio_escola"].fillna("Não informado")
)

laboratorio_escola,Não há computadores ou recursos tecnológicos disponíveis,"Não há laboratório, mas há alguns computadores disponíveis",Não informado,"Sim, há laboratório de informática bem equipado","Sim, mas os equipamentos são insuficientes ou desatualizados"
grupo,,,,,
EB,0,0,74,0,0
GRAD,0,0,37,0,0
POS,0,0,5,0,0
PROF_EB,2,3,0,7,2


In [96]:
mapeamento_laboratorio = {
    "Sim, há laboratório de informática bem equipado":
        "Infraestrutura adequada",

    "Sim, mas os equipamentos são insuficientes ou desatualizados":
        "Infraestrutura limitada",

    "Não há laboratório, mas há alguns computadores disponíveis":
        "Infraestrutura limitada",

    "Não há computadores ou recursos tecnológicos disponíveis":
        "Infraestrutura ausente"
}

df_perfil_limpo["laboratorio_categoria"] = (
    df_perfil_limpo["laboratorio_escola"]
    .map(mapeamento_laboratorio)
)

In [97]:
df_perfil_limpo["laboratorio_categoria"].value_counts(
    dropna=False
)

laboratorio_categoria
NaN                        116
Infraestrutura adequada      7
Infraestrutura limitada      5
Infraestrutura ausente       2
Name: count, dtype: int64

In [98]:
df_perfil_limpo["acesso_estudantes_tecnologia"].value_counts(
    dropna=False
)

acesso_estudantes_tecnologia
NaN                                                   116
Alguns têm acesso, mas de forma limitada                8
Todos têm acesso regular a computadores e internet      5
Poucos têm acesso                                       1
Name: count, dtype: int64

In [99]:
pd.crosstab(
    df_perfil_limpo["grupo"],
    df_perfil_limpo["acesso_estudantes_tecnologia"].fillna("Não informado")
)

acesso_estudantes_tecnologia,"Alguns têm acesso, mas de forma limitada",Não informado,Poucos têm acesso,Todos têm acesso regular a computadores e internet
grupo,,,,
EB,0,74,0,0
GRAD,0,37,0,0
POS,0,5,0,0
PROF_EB,8,0,1,5


In [100]:
df_perfil_limpo["cidade_uf"].value_counts(
    dropna=False
)

cidade_uf
Florianópolis - Santa Catarina    21
Niterói - Rio de Janeiro          20
Belém - Pará                      16
Juiz de Fora - Minas Gerais       15
Russas - Ceará                    13
João Pessoa - Paraíba             11
Ceres - Goiás                     10
Manaus - Amazonas                  8
Santa Maria/DF                     6
Palhano - Ceará                    1
São Gonçalo - Rio de Janeiro       1
Itapaci-Goiás                      1
Samambaia sul                      1
Rio Tinto- Paraíba                 1
Brasília - distrito federal        1
Sobradinho - DF                    1
Limoeiro do Norte - Ceará          1
Taubaté-SP                         1
Rio de Janeiro/ RJ                 1
Name: count, dtype: int64

In [101]:
pd.crosstab(
    df_perfil_limpo["grupo"],
    df_perfil_limpo["cidade_uf"].notna()
)

cidade_uf,True
grupo,
EB,74
GRAD,37
POS,5
PROF_EB,14


In [103]:
mapeamento_localizacao = {
    "Florianópolis - Santa Catarina": ("Florianópolis", "SC"),
    "Niterói - Rio de Janeiro": ("Niterói", "RJ"),
    "Belém - Pará": ("Belém", "PA"),
    "Juiz de Fora - Minas Gerais": ("Juiz de Fora", "MG"),
    "Russas - Ceará": ("Russas", "CE"),
    "João Pessoa - Paraíba": ("João Pessoa", "PB"),
    "Ceres - Goiás": ("Ceres", "GO"),
    "Manaus - Amazonas": ("Manaus", "AM"),
    "Santa Maria/DF": ("Santa Maria", "DF"),
    "Palhano - Ceará": ("Palhano", "CE"),
    "São Gonçalo - Rio de Janeiro": ("São Gonçalo", "RJ"),
    "Itapaci-Goiás": ("Itapaci", "GO"),
    "Samambaia sul": ("Samambaia Sul", "DF"),
    "Rio Tinto- Paraíba": ("Rio Tinto", "PB"),
    "Brasília - distrito federal": ("Brasília", "DF"),
    "Sobradinho - DF": ("Sobradinho", "DF"),
    "Limoeiro do Norte - Ceará": ("Limoeiro do Norte", "CE"),
    "Taubaté-SP": ("Taubaté", "SP"),
    "Rio de Janeiro / RJ": ("Rio de Janeiro", "RJ")
}

df_perfil_limpo["cidade_participante"] = (
    df_perfil_limpo["cidade_uf"]
    .map(lambda x: mapeamento_localizacao.get(x, (pd.NA, pd.NA))[0])
)

df_perfil_limpo["estado_participante"] = (
    df_perfil_limpo["cidade_uf"]
    .map(lambda x: mapeamento_localizacao.get(x, (pd.NA, pd.NA))[1])
)

In [104]:
df_perfil_limpo["regiao_participante"] = (
    df_perfil_limpo["estado_participante"]
    .map(regioes)
)

In [105]:
print("Total de participantes:", len(df_perfil_limpo))

print(
    "Cidades identificadas:",
    df_perfil_limpo["cidade_participante"].notna().sum()
)

print(
    "Estados identificados:",
    df_perfil_limpo["estado_participante"].notna().sum()
)

print(
    "Regiões identificadas:",
    df_perfil_limpo["regiao_participante"].notna().sum()
)

display(
    df_perfil_limpo["estado_participante"]
    .value_counts(dropna=False)
)

Total de participantes: 130
Cidades identificadas: 129
Estados identificados: 129
Regiões identificadas: 129


estado_participante
SC     21
RJ     21
PA     16
CE     15
MG     15
PB     12
GO     11
DF      9
AM      8
SP      1
NaN     1
Name: count, dtype: int64

In [106]:
display(
    df_perfil_limpo.loc[
        df_perfil_limpo["estado_participante"].isna(),
        ["cidade_uf"]
    ].drop_duplicates()
)

,cidade_uf
118,Rio de Janeiro/ RJ


In [107]:
from pathlib import Path

PASTA_PROCESSED = PROJETO / "data" / "processed"
PASTA_PROCESSED.mkdir(parents=True, exist_ok=True)

CAMINHO_PERFIL = PASTA_PROCESSED / "dashboard_perfil.xlsx"

df_perfil_limpo.to_excel(
    CAMINHO_PERFIL,
    sheet_name="Perfil_Participantes",
    index=False,
    engine="openpyxl"
)

print("Arquivo salvo em:", CAMINHO_PERFIL)

Arquivo salvo em: /home/mjulia18/base_de_dados_aberta/renacee-avaliacao-anual/data/processed/dashboard_perfil.xlsx


In [108]:
df_conferencia = pd.read_excel(CAMINHO_PERFIL)

print("Linhas:", df_conferencia.shape[0])
print("Colunas:", df_conferencia.shape[1])

print(
    "IDs duplicados:",
    df_conferencia["ID"].duplicated().sum()
)

Linhas: 130
Colunas: 53
IDs duplicados: 0
